# INT395: Supervised Learning - Course Project
## Project 15: Campus Energy Consumption Intelligence & Anomaly Detection

**Objective**: Develop an end-to-end intelligent machine learning platform for analyzing and predicting electricity consumption across university campus facilities, identifying normal vs. abnormal consumption patterns, detecting energy wastage (e.g. off-hours HVAC runaways, equipment faults), and providing explainable insights through SHAP.

### Complete ML Lifecycle:
$$\text{Problem} \to \text{Data Acquisition} \to \text{EDA} \to \text{Preprocessing} \to \text{Feature Engineering} \to \text{Model Zoo} \to \text{Optimization} \to \text{Explainability (SHAP)} \to \text{Anomaly Detection} \to \text{Deployment}$$

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Import project source modules
sys.path.insert(0, '..')
from src.data_generator import generate_campus_dataset, BUILDING_PROFILES
from src.preprocessing import engineer_features, get_feature_columns, train_val_test_split
from src.models import train_and_evaluate_all, calculate_metrics
from src.anomaly_detector import EnergyAnomalyDetector, evaluate_anomaly_detection
from src.explainability import EnergyExplainer

## 1. Dataset Generation & Data Quality Inspection
We generate high-resolution hourly campus energy data spanning 2 years across 5 key university buildings:
1. `Academic_Block_1` (Lecture halls, computing labs, faculty offices)
2. `Central_Library` (24x7 study zones, reading rooms, air-conditioned stacks)
3. `Hostel_Girls_Block` & `Hostel_Boys_Block` (Residential dormitories)
4. `Dining_Hall_Mess` (Central dining facility with meal surge cycles)

In [ ]:
# Generate 2-year hourly dataset
df_raw = generate_campus_dataset(start_date="2024-01-01", end_date="2025-12-31", seed=42)
print(f"Total records: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 2. Exploratory Data Analysis (EDA)
Let us examine:
- Diurnal 24-hour profiles by building type
- Weekday vs Weekend energy variation
- Temperature and thermal load correlations
- Sub-metering load distribution (HVAC, Lighting, Equipment)

In [ ]:
# Diurnal profile comparison across building types
plt.figure(figsize=(14, 5))
sns.lineplot(data=df_raw, x='hour', y='total_power_kwh', hue='building_id', ci=None, linewidth=2.5)
plt.title('Campus Diurnal Electricity Demand Profile by Building (24h Cycle)', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day (0 - 23)')
plt.ylabel('Electricity Consumption (kWh)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Outdoor Temperature vs Building Electricity Demand
plt.figure(figsize=(12, 5))
sns.scatterplot(data=df_raw.sample(2500, random_state=42), x='temperature_c', y='total_power_kwh', hue='building_type', alpha=0.6)
plt.title('Electricity Consumption (kWh) vs Outdoor Ambient Temperature (°C)', fontsize=14, fontweight='bold')
plt.xlabel('Temperature (°C)')
plt.ylabel('Total Power (kWh)')
plt.tight_layout()
plt.show()

## 3. Preprocessing & Feature Engineering
We engineer:
- **Cyclic Time Transforms**: $\sin/\cos$ mappings for hour, day of week, and month
- **Weather Indices**: Cooling Degree Hours (CDH), Heating Degree Hours (HDH), Heat Index
- **Lag Features**: $t-1h, t-2h, t-24h, t-48h, t-168h$ (1 week)
- **Rolling Statistics**: 6h, 24h, 168h rolling mean, std, max, min

In [ ]:
df_featured = engineer_features(df_raw)
feature_cols, target_col = get_feature_columns()
print(f"Total features engineered: {len(feature_cols)}")
print("Engineered Features:", feature_cols)

# Chronological Train/Val/Test Split (Preventing Data Leakage)
train_df, val_df, test_df = train_val_test_split(df_featured, train_ratio=0.70, val_ratio=0.15)
print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,} | Test samples: {len(test_df):,}")

## 4. Supervised Model Zoo Development & Benchmarking
We train and benchmark:
1. **Persistence Baseline** (Lag-24 baseline)
2. **Ridge Regression** (L2 Linear Baseline)
3. **Random Forest Regressor** (Bagged Trees)
4. **LightGBM Regressor** (Histogram-based GBDT)
5. **XGBoost Regressor** (Extreme Gradient Boosting)
6. **Stacking Ensemble** (Multi-model blending via Ridge Meta-Regressor)

In [ ]:
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

trained_models, comparison_df, test_preds = train_and_evaluate_all(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("\n--- MODEL COMPARISON LEADERBOARD ---")
display(comparison_df)

## 5. Model Evaluation & Residual Error Analysis

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
best_preds = test_preds[best_model_name]
residuals = y_test - best_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(residuals, kde=True, ax=axes[0], color='#4F46E5', bins=40)
axes[0].set_title(f'Residual Error Distribution ({best_model_name})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Prediction Error e = y - ŷ (kWh)')

axes[1].scatter(best_preds, y_test, alpha=0.3, color='#06B6D4', edgecolors='none', s=20)
min_v, max_v = min(best_preds.min(), y_test.min()), max(best_preds.max(), y_test.max())
axes[1].plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Perfect Fit Line (y=x)')
axes[1].set_title('Predicted vs Actual Consumption (Goodness-of-Fit)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Load (kWh)')
axes[1].set_ylabel('Actual Load (kWh)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Explainable AI (SHAP - SHapley Additive exPlanations)
We investigate why the model produces particular predictions:
- **Global Feature Importance**: Ranking the most critical drivers across all campus buildings
- **Local Waterfall Explanations**: Decomposing individual hourly forecasts into additive impacts

In [ ]:
tree_model = trained_models.get('XGBoost Regressor', trained_models[best_model_name])
explainer = EnergyExplainer(tree_model, feature_cols)
shap_vals, importance_df = explainer.explain_global(X_test, max_samples=400)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='Mean_Abs_SHAP', y='Feature', palette='Blues_r')
plt.title('Global SHAP Feature Importance Ranking', fontsize=14, fontweight='bold')
plt.xlabel('Mean |SHAP Value| (Average Impact on Prediction in kWh)')
plt.tight_layout()
plt.show()

## 7. Anomaly Detection, Energy Wastage & Risk Scoring
We use **Isolation Forest** coupled with **Residual Outlier Detection** to flag unexpected energy spikes, equipment overloads, and off-hours leaks.

In [ ]:
detector = EnergyAnomalyDetector(contamination=0.03)
detector.fit(train_df)
test_anoms = detector.detect_anomalies(test_df, best_preds)
anom_metrics = evaluate_anomaly_detection(test_anoms)

print("--- ANOMALY DETECTION EVALUATION ---")
for k, v in anom_metrics.items():
    print(f"{k}: {v}")

print("\n--- ACTIONABLE ANOMALIES SAMPLE ---")
display(test_anoms[test_anoms['detected_anomaly'] == 1][[
    'timestamp', 'building_id', 'detected_severity', 'total_power_kwh', 'forecast_kwh',
    'wasted_energy_kwh', 'financial_loss_inr', 'root_cause'
]].head(10))

## 8. Summary & Deployment Readiness
- All models, preprocessing transformers, and anomaly engines are serialized into `saved_models/`.
- The interactive Streamlit dashboard (`app.py`) provides live real-time monitoring, what-if simulations, and XAI visualizers for campus facility managers.